In [ ]:
!pip install transformers torch torchvision accelerate

In [2]:
import os
from transformers import Pix2StructProcessor, Pix2StructForConditionalGeneration
from PIL import Image
from glob import glob

print("1. Memuat Model DePlot... (Mungkin butuh waktu beberapa menit untuk unduhan awal)")
# Menggunakan model DePlot dari Google
processor = Pix2StructProcessor.from_pretrained('google/deplot')
model = Pix2StructForConditionalGeneration.from_pretrained('google/deplot')

# Pindahkan model ke GPU jika tersedia (opsional tapi sangat disarankan)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model berhasil dimuat menggunakan perangkat: {device}")

# 2. Ambil satu sampel gambar dari synthetic_data
synthetic_dir = '../data/raw/synthetic_data'
sample_img_path = glob(os.path.join(synthetic_dir, 'charts', '*.*'))[0]
image = Image.open(sample_img_path)

# Tampilkan gambar yang sedang diproses
display(image.resize((400, int(400 * image.size[1] / image.size[0])))) # Resize untuk preview

# 3. Proses Gambar (Inference)
print(f"\n2. Mengekstrak teks dari gambar: {os.path.basename(sample_img_path)}...")
# Prompt bawaan DePlot untuk ekstraksi chart
inputs = processor(images=image, text="Generate underlying data table of the figure below:", return_tensors="pt").to(device)

# max_new_tokens diset 512 agar cukup untuk tabel yang panjang
predictions = model.generate(**inputs, max_new_tokens=512)
extracted_text = processor.decode(predictions[0], skip_special_tokens=True)

# 4. Tampilkan Hasil Ekstraksi
print("\n=== HASIL EKSTRAKSI DEPLOT (VISION MODEL) ===")
# DePlot biasanya memisahkan kolom dengan " | " dan baris dengan "<0x0A>"
formatted_text = extracted_text.replace("<0x0A>", "\n")
print(formatted_text)

# 5. Bandingkan dengan Ground Truth (Asli)
base_name = os.path.splitext(os.path.basename(sample_img_path))[0]
md_path = os.path.join(synthetic_dir, 'markdown', f"{base_name}.md")

if os.path.exists(md_path):
    with open(md_path, 'r', encoding='utf-8') as f:
        print("\n=== GROUND TRUTH (DATA ASLI DARI KAGGLE) ===")
        print(f.read()[:800])
else:
    print("\nFile Markdown asli tidak ditemukan.")

1. Memuat Model DePlot... (Mungkin butuh waktu beberapa menit untuk unduhan awal)


D:\latihan\analisis grafik\venv\lib\site-packages\huggingface_hub\file_download.py:746: UserWarning: Not enough free disk space to download the file. The expected file size is: 1129.18 MB. The target location C:\Users\Pongo\.cache\huggingface\hub\models--google--deplot\blobs only has 740.90 MB free disk space.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.13GB            

model.safetensors: downloading bytes:           |  0.00B            

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

